<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/midterm-project/notebooks/AI301_Midterm_Guided_Lab_Meet_the_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Midterm Guided Lab · Meet the Dataset

**Midterm Project · Medical Abstracts**

> Save a copy in your Drive before you start (*File > Save a copy in Drive*). The default CPU runtime is enough. The notebook downloads the Medical Abstracts dataset from Hugging Face, about 14,000 abstracts of text, and in Part 4 one small embedding model, gte-small.

The midterm runs on the Medical Abstracts dataset. This notebook looks at it before any model does: what one row holds, how the five conditions are spread, and two problems in the raw files that the midterm notebook fixes in its cleaning step. Each problem is shown first on one real abstract, then counted over the whole dataset.

## Part 1 · Loading the data

> **Dataset card:** [TimSchopf/medical_abstracts](https://huggingface.co/datasets/TimSchopf/medical_abstracts). Read it for the five condition labels and the paper the dataset comes from. The authors also keep the files on [GitHub](https://github.com/sebischair/Medical-Abstracts-TC-Corpus).

The dataset comes in two splits, `train` and `test`. We load each one, mark which split every row came from, and put them in one table so we can compare them.

- **Python note:** `load_dataset(name, split="train")` downloads one split. `.to_pandas()` turns it into a pandas DataFrame, a table with named columns.
- **Python note:** `df["split"] = "train"` adds a column with the same value on every row. `df["row"] = df.index` keeps each row's position in its own split file, so we can find it again later.
- **Python note:** `pd.concat([a, b])` stacks two tables with the same columns, one under the other.

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # tqdm warns about ipywidgets outside Colab; it is not an error

import pandas as pd
from datasets import load_dataset

train_df = load_dataset("TimSchopf/medical_abstracts", split="train").to_pandas()
test_df = load_dataset("TimSchopf/medical_abstracts", split="test").to_pandas()

train_df["split"] = "train"
train_df["row"] = train_df.index
test_df["split"] = "test"
test_df["row"] = test_df.index

df = pd.concat([train_df, test_df], ignore_index=True)

print("train rows:", len(train_df))
print("test rows:", len(test_df))
print("all rows:", len(df))

11,550 training rows and 2,888 test rows, 14,438 in all.

## Part 2 · First look

Each row has two columns from the dataset: `medical_abstract`, the title followed by the abstract text, and `condition_label`, a number from 1 to 5. The dataset card gives the name of each number. We store the names in a dictionary and add them as a column.

| Number | Condition |
|---|---|
| 1 | neoplasms (tumors and cancer) |
| 2 | digestive system diseases |
| 3 | nervous system diseases |
| 4 | cardiovascular diseases |
| 5 | general pathological conditions |

- **Python note:** `LABEL_NAMES[v]` looks up the name for one number. The loop builds the list of names one row at a time, then the list becomes a new column.

In [ ]:
LABEL_NAMES = {
    1: "neoplasms",
    2: "digestive system diseases",
    3: "nervous system diseases",
    4: "cardiovascular diseases",
    5: "general pathological conditions",
}

names = []
for v in df["condition_label"]:
    names.append(LABEL_NAMES[int(v)])
df["label_name"] = names

df.head()

Read one abstract in full before looking at counts.

In [ ]:
first = df.iloc[0]
print("split:", first["split"], " label:", first["label_name"])
print()
print(first["medical_abstract"])

## Part 3 · The five conditions

Four of the labels name an organ system or a disease group. The fifth, general pathological conditions, is for abstracts that do not belong to one of the other four.



**Your prediction:**

_Type here._

In [ ]:
import matplotlib.pyplot as plt

counts = df["label_name"].value_counts()

rows = []
for name in counts.index:
    rows.append({"condition": name, "rows": counts[name], "percent": round(counts[name] / len(df) * 100, 1)})
display(pd.DataFrame(rows))

plt.figure(figsize=(8, 4))
plt.barh(counts.index, counts.values, color="#FF8200")
plt.xlabel("Rows")
plt.title("Rows per condition, train and test together")
plt.gca().invert_yaxis()
plt.show()

General pathological conditions has the most rows, 4,805 (33.3 percent), and digestive system diseases the fewest, 1,494 (10.3 percent). A model that ignores the text and always answers general pathological conditions would be right about one time in three.

**Interpret the output.**

1. The largest label is the one defined by what it is not (geneal conditions). What does that suggest about how hard it will be to classify?

**Your answer:**

_Type here._

## Part 4 · Abstract length

We have not said much yet about **context windows**. Every encoder has one: the largest number of tokens it reads at once. Whichever embedding model you choose for the midterm, check its window on the model card before you use it.

In the midterm, an embedding model (gte-small) turns each abstract into one vector. First its tokenizer splits the abstract into tokens and adds two special tokens, one at the start and one at the end. The model reads at most 512 tokens, so about 510 tokens of the abstract fit. If an abstract is longer, the rest is dropped before the model runs: it is not read, so the end of that abstract plays no part in its vector.

Medical terms split into many pieces: in Module 2, *cholecystectomy* became six tokens. With a BERT tokenizer these abstracts came to about 1.6 tokens per word, so an abstract of roughly 315 words reaches the limit. gte-small has its own tokenizer, so treat 315 as an estimate.

Counting words needs no model, so it is a quick first check of how many abstracts are long enough to be cut.

- **Python note:** `.str.split().str.len()` splits each text on spaces and counts the pieces, a quick word count. `.describe()` prints the count, mean, spread, minimum, quartiles and maximum of a column.
- **Python note:** `(df["words"] > 315).sum()` marks each abstract `True` if it has more than 315 words and counts the `True` values.

In [ ]:
df["words"] = df["medical_abstract"].str.split().str.len()
print(df["words"].describe().round(1))
print()

long_count = (df["words"] > 315).sum()
print("abstracts over 315 words:", long_count, round(long_count / len(df) * 100, 1), "percent")
print()

rows = []
for name in LABEL_NAMES.values():
    rows.append({"condition": name, "median words": df[df["label_name"] == name]["words"].median()})
display(pd.DataFrame(rows))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# left: how long the abstracts are, with the estimated cut point
axes[0].hist(df["words"], bins=60, color="#FF8200")
axes[0].axvline(315, color="#58595B", linestyle="--", linewidth=2, label="about 512 tokens (315 words)")
axes[0].set_xlabel("Words per abstract")
axes[0].set_ylabel("Number of abstracts")
axes[0].set_title("Abstract length")
axes[0].legend()

# right: one box per condition
lengths = []
for name in LABEL_NAMES.values():
    lengths.append(df[df["label_name"] == name]["words"])
axes[1].boxplot(lengths, vert=False)
axes[1].set_yticklabels(list(LABEL_NAMES.values()))
axes[1].axvline(315, color="#58595B", linestyle="--", linewidth=2)
axes[1].set_xlabel("Words per abstract")
axes[1].set_title("Length by condition")

plt.tight_layout()
plt.show()

A median of 176 words and a longest abstract of 596. 569 abstracts, 3.9 percent, have more than 315 words, so they are the ones likely to lose their end. Cardiovascular abstracts are the longest (median 205 words) and nervous system the shortest (157).

### What happens past the limit

> **Model card:** [thenlper/gte-small](https://huggingface.co/thenlper/gte-small). Read it for the maximum input length and the size of the vectors. This cell downloads the model, a small one.

A small test shows the cut. We build three made up texts from two words that are one token each:

| Text | What it holds | Where *cancer* falls |
|---|---|---|
| long | "heart" 600 times, then "cancer" 400 times | after the limit |
| cut | "heart" 600 times only | nowhere |
| cancer first | "cancer" 400 times, then "heart" 600 times | inside the limit |

We encode all three and compare them with cosine similarity, as in M04 Guided Lab Part 2. A value of 1.0 means two vectors point the same way; lower values mean they differ.

- **Python note:** `model.encode(a_list)` returns one row of 384 numbers per text. `vectors[0][:8]` is the first 8 numbers of the first vector.
- **Python note:** `cosine_similarity` takes two lists of vectors and returns a table of scores, so each vector is wrapped in `[ ]` and the single score is read with `[0][0]`.

> **Predict before running.** What will the similarity of long and cut be? And of long and cancer first?

**Your prediction:**

_Type here._

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("thenlper/gte-small")
print("max input length:", model.max_seq_length)

texts = {
    "long (cancer after limit)": "heart " * 600 + "cancer " * 400,
    "cut (heart only)": "heart " * 600,
    "cancer first": "cancer " * 400 + "heart " * 600,
}

names = []
for name in texts:
    names.append(name)

vectors = model.encode(list(texts.values()))
print("shape:", vectors.shape)   # 3 texts, 384 numbers each
print()

for i in range(len(names)):
    print(names[i], np.round(vectors[i][:8], 3))
print()

print("long vs cut:", cosine_similarity([vectors[0]], [vectors[1]])[0][0])
print("long vs cancer first:", cosine_similarity([vectors[0]], [vectors[2]])[0][0])

**Interpret the output.**

1. Compare the first 8 numbers of long and cut. What does that tell you about the 400 *cancer* tokens in the long text?
2. Why is long against cancer first lower, when both texts contain the same words?
3. About 4 percent of the abstracts are long enough to be cut. Should that change how you read the midterm results?

**Your answer:**

_Type here._

## Part 5 · One abstract, two labels

The file has one label per row. When the authors gave an abstract more than one condition, they wrote it once per condition, so the same text appears on two or more rows with different labels. No single row shows this. It shows only when you look for the same text twice.

pandas has a method for exactly that, `duplicated`.

- **Python note:** `df["medical_abstract"].duplicated(keep=False)` marks a row `True` if its text appears on any other row. `keep=False` marks every copy; the default marks only the second and later copies.
- **Python note:** `df.duplicated(subset=[a, b])` marks a row `True` only if another row has the same values in both columns, here the same text and the same label.
- **Python note:** `.str[:80]` keeps the first 80 characters of each text, so the abstract fits in the table. `sort_values` puts rows with the same text next to each other.
- **Python note:** `pd.set_option("display.max_colwidth", 80)` lets a table cell show up to 80 characters instead of the default 50.

In [ ]:
pd.set_option("display.max_colwidth", 80)   # show up to 80 characters of text in a table cell

repeated = df[df["medical_abstract"].duplicated(keep=False)].copy()
print("rows whose text appears on another row:", len(repeated))
print("rows that repeat both the text and the label:", df.duplicated(subset=["medical_abstract", "condition_label"]).sum())

repeated["abstract start"] = repeated["medical_abstract"].str[:80]
repeated = repeated.sort_values(["medical_abstract", "condition_label"])
display(repeated[["abstract start", "split", "row", "label_name"]].head(10))

6,140 rows have text that appears on another row, and none of them repeats both the text and the label. Every repeat is the same abstract under a different condition. In the table, rows with the same start are one abstract.

Here is one of them in full, an abstract that begins "7th nerve palsy".

- **Python note:** `.str.startswith(text)` marks each row `True` if the abstract begins with that text.

In [ ]:
pd.set_option("display.max_colwidth", 80)   # show up to 80 characters of text in a table cell

example = df[df["medical_abstract"].str.startswith("7th nerve palsy")].copy()
example["abstract start"] = example["medical_abstract"].str[:80]
display(example[["abstract start", "split", "row", "condition_label", "label_name"]])

print(example["medical_abstract"].iloc[0])

Two rows, the same text. One is in the training split labeled nervous system diseases, the other in the test split labeled general pathological conditions. Neither label is wrong: a palsy of a cranial nerve is a nervous system problem, and a complication after a medical procedure fits the general group.



`duplicated` finds repeated text, one row at a time. To count abstracts rather than rows, `groupby` collects the rows that share the same text, and `nunique` counts how many different labels each text has. Any count above 1 is an abstract with more than one label, called a **multi-label** abstract.

- **Python note:** `df.groupby(column_a)[column_b].nunique()` returns one number per distinct value of `column_a`: how many different values of `column_b` appear with it.
- **Python note:** `n_labels[n_labels > 1]` keeps only the entries above 1. Its `.index` is the list of those texts.

In [ ]:
n_labels = df.groupby("medical_abstract")["condition_label"].nunique()
multi = n_labels[n_labels > 1]

print("distinct abstracts:", len(n_labels))
print("abstracts with more than one label:", len(multi))
print("rows those abstracts take up:", df["medical_abstract"].isin(multi.index).sum())
print()
print("how many labels each has:")
print(multi.value_counts().sort_index())

Of 11,227 distinct abstracts, 2,929 have more than one label, and they take up 6,140 of the 14,438 rows. Most have two labels; 270 have three and 6 have four.

Which labels go together? The next cell writes the labels of each multi-label abstract as one string, such as "nervous system diseases + general pathological conditions", and counts the strings.

In [ ]:
pairs = []
for text in multi.index:
    labels_here = sorted(df[df["medical_abstract"] == text]["label_name"].unique())
    pairs.append(" + ".join(labels_here))

pd.Series(pairs).value_counts().head(10)

The four most common combinations each join an organ label to general pathological conditions: cardiovascular (738), neoplasms (486), digestive (475) and nervous system (446). The catch-all label overlaps with every other one.

Three more examples, each with every row it appears on.

In [ ]:
shown = 0
for text in multi.index:
    rows_here = df[df["medical_abstract"] == text]
    display(rows_here[["split", "row", "label_name"]])
    print(text[:250])
    print()
    shown = shown + 1
    if shown == 3:
        break

**Interpret the output.**

1. Read the three examples. For each, is one of the labels a mistake, or does the abstract fit both?
2. A classifier in the midterm predicts one condition per abstract. What should count as the right answer for an abstract with two labels?

**Your answer:**

_Type here._

## Part 6 · The same abstract in train and test

The 7th nerve palsy abstract had one row in each split. A model trained on the training split would learn that text as nervous system, then be tested on the very same text and marked wrong unless it says general pathological conditions.



- **Python note:** `set(...)` keeps each text once and makes lookups fast. `.isin(a_set)` marks each row `True` if its text is in the set.
- **Python note:** `merge` joins two tables on a shared column, here the abstract text, so each matching pair of rows becomes one row with both labels side by side. `suffixes` names the two label columns.

In [ ]:
train_texts = set(train_df["medical_abstract"])
in_both = test_df[test_df["medical_abstract"].isin(train_texts)]
print("test rows whose text is also in train:", len(in_both))
print("distinct abstracts:", in_both["medical_abstract"].nunique())

matched = test_df.merge(train_df, on="medical_abstract", suffixes=("_test", "_train"))
same = 0
for i in range(len(matched)):
    if matched["condition_label_test"].iloc[i] == matched["condition_label_train"].iloc[i]:
        same = same + 1
print("pairs with the same label in both splits:", same)

1,010 test rows, 988 distinct abstracts, also appear in training, and none has the same label in both splits. Every one is a multi-label abstract whose rows landed in different splits. The overlap and the multi-label rows are the same problem seen from two sides.

## Part 7 · What cleaning keeps

The midterm treats the task as single-label: one condition per abstract. So we will drop every multi-label abstract, all of its rows. This is a choice, and it has a cost.

> **Predict before running.** Which condition will lose the largest share of its rows?

**Your prediction:**

_Type here._

- **Python note:** `~` flips `True` and `False`, so `df[~df["medical_abstract"].isin(multi.index)]` keeps the rows whose text is not multi-label.

In [ ]:
clean = df[~df["medical_abstract"].isin(multi.index)]
print("rows before:", len(df), " after:", len(clean))
print("repeated texts left:", clean["medical_abstract"].duplicated().sum())
print()

before = df["label_name"].value_counts()
after = clean["label_name"].value_counts()
rows = []
for name in LABEL_NAMES.values():
    rows.append({"condition": name, "before": before[name], "after": after[name],
                 "percent kept": round(after[name] / before[name] * 100),
                 "share after": round(after[name] / len(clean) * 100, 1)})
display(pd.DataFrame(rows))

clean_train = set(clean[clean["split"] == "train"]["medical_abstract"])
clean_test = set(clean[clean["split"] == "test"]["medical_abstract"])
print("train abstracts:", len(clean_train), " test abstracts:", len(clean_test))
print("in both after cleaning:", len(clean_train & clean_test))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# conditions in order of size before cleaning, largest at the top
conditions = list(before.sort_values(ascending=False).index)

kept = []
dropped = []
share_before = []
share_after = []
for name in conditions:
    kept.append(after[name])
    dropped.append(before[name] - after[name])
    share_before.append(before[name] / len(df) * 100)
    share_after.append(after[name] / len(clean) * 100)

y = np.arange(len(conditions))
ORANGE = "#FF8200"
GRAY = "#C9C9C9"
INK = "#58595B"

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [3, 2]})

# left: one bar per condition, kept plus dropped = rows before cleaning
axes[0].barh(y, kept, height=0.6, color=ORANGE, edgecolor="white", linewidth=2, label="kept")
axes[0].barh(y, dropped, left=kept, height=0.6, color=GRAY, edgecolor="white", linewidth=2, label="dropped (multi-label)")
for i in range(len(conditions)):
    total = kept[i] + dropped[i]
    percent = round(kept[i] / total * 100)
    label = str(kept[i]) + " of " + str(total) + " kept (" + str(percent) + "%)"
    axes[0].text(total + 60, y[i], label, va="center", color=INK, fontsize=9)
axes[0].set_yticks(y)
axes[0].set_yticklabels(conditions)
axes[0].invert_yaxis()
axes[0].set_xlim(0, max(before) * 1.45)
axes[0].set_xlabel("Rows")
axes[0].set_title("Rows kept and dropped by cleaning", loc="left")
axes[0].legend(loc="lower right", frameon=False)

# right: share of the data before (gray) and after (orange), joined by a line
for i in range(len(conditions)):
    axes[1].plot([share_before[i], share_after[i]], [y[i], y[i]], color=GRAY, linewidth=2, zorder=1)
axes[1].scatter(share_before, y, s=70, color=GRAY, edgecolor="white", linewidth=2, zorder=2, label="before")
axes[1].scatter(share_after, y, s=70, color=ORANGE, edgecolor="white", linewidth=2, zorder=3, label="after")
for i in range(len(conditions)):
    axes[1].text(share_after[i], y[i] - 0.3, str(round(share_after[i], 1)) + "%", ha="center", color=INK, fontsize=9)
axes[1].set_yticks(y)
axes[1].set_yticklabels([])
axes[1].invert_yaxis()
axes[1].set_xlim(0, 40)
axes[1].set_xlabel("Percent of all rows")
axes[1].set_title("Share of the data", loc="left")
axes[1].legend(loc="lower right", frameon=False)

# quiet the frame: no top or right border, light vertical grid behind the marks
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#EEEEEE")
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### What we have after cleaning

One table of the data the midterm starts from: each condition's abstracts, its share of the cleaned data, and how many sit in each split.

- **Python note:** `(part["split"] == "train").sum()` marks each row `True` if it is in the training split and counts the `True` values.

In [ ]:
rows = []
for name in conditions:
    part = clean[clean["label_name"] == name]
    rows.append({"condition": name,
                 "abstracts": len(part),
                 "share (%)": round(len(part) / len(clean) * 100, 1),
                 "train": (part["split"] == "train").sum(),
                 "test": (part["split"] == "test").sum()})
rows.append({"condition": "total",
             "abstracts": len(clean),
             "share (%)": 100.0,
             "train": (clean["split"] == "train").sum(),
             "test": (clean["split"] == "test").sum()})
display(pd.DataFrame(rows))

8,298 abstracts remain, each with one label and no repeats. The train and test splits no longer share any text: 6,607 and 1,691.

General pathological conditions keeps only half its rows, 2,394 of 4,805, and digestive keeps 47 percent. Neoplasms keeps the most, 69 percent. The order of the classes also changes: digestive drops to 8.4 percent of the data.

**Interpret the output.**

1. The dropped abstracts are the ones that sit between two conditions. Will scores on the cleaned data be higher or lower than on data that kept them, and why?
2. Name one other way to handle a multi-label abstract instead of dropping it, and what it would cost.

**Your answer:**

_Type here._

## Part 8 · What the labels are and are not

- **Where they come from.** The authors started from a [Kaggle corpus](https://www.kaggle.com/datasets/chaitanyakck/medical-text) of 28,880 abstracts, of which about half had a label. They kept the labeled half and gave the numbers names (Schopf, Braun and Matthes, 2022).
- **What a label is.** The condition the dataset authors assigned to an abstract.
- **What a label is not.** It is not a diagnosis of any patient. It is not a full description of the paper: Parts 5 to 7 showed that many papers fit more than one condition. And the counts are not a measure of how common a disease is. They show which abstracts were collected, which had a label, and which survived cleaning.

## Check yourself

- Say why an abstract with two labels cannot be seen on any single row of the file.
- Say why one abstract appearing in both train and test is a problem for testing a model.
- Say which condition cleaning hits hardest, and why.

## Summary

The dataset has 14,438 rows and five condition labels, with general pathological conditions the largest. 2,929 abstracts have more than one label and are stored once per label, so the same text appears on several rows. Those rows are also the only text shared by the train and test splits. Dropping them leaves 8,298 single-label abstracts with no overlap, at the cost of half the general pathological conditions rows. The midterm notebook starts from that cleaned table.

## Appendix · Keeping the end of a long abstract, optional

One way to keep the part past the limit, used by the dataset paper (Schopf, Braun and Matthes, 2022, Algorithm 1):

1. Split the abstract into pieces that each fit inside the limit.
2. Encode every piece.
3. Average the piece vectors into one vector and normalize it again.

The cost is one extra encode for each extra piece. A piece can end in the middle of a sentence, and a short last piece counts as much as a full one in the average. Two other ways: keep the first and the last part and drop the middle, or use a model with a longer context window (check its card).

- **Python note:** `model.tokenizer(text, add_special_tokens=False)["input_ids"]` gives the token ids of the text without the two special tokens. `range(0, len(ids), CHUNK)` steps through them 500 at a time, and `tokenizer.decode` turns each slice back into text.
- **Python note:** `np.linalg.norm(mean)` is the length of the averaged vector. Dividing by it gives the vector length 1, as `normalize_embeddings=True` does.

In [ ]:
tokenizer = model.tokenizer
CHUNK = 500     # tokens per piece, under the 510 that fit


def encode_long(text):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    pieces = []
    for start in range(0, len(ids), CHUNK):
        piece_ids = ids[start:start + CHUNK]
        pieces.append(tokenizer.decode(piece_ids))
    piece_vectors = model.encode(pieces, normalize_embeddings=True)
    mean = piece_vectors.mean(axis=0)
    return mean / np.linalg.norm(mean)


v_long = encode_long(texts["long (cancer after limit)"])
v_cut = model.encode(texts["cut (heart only)"], normalize_embeddings=True)
print("chunked long vs cut:", cosine_similarity([v_long], [v_cut])[0][0])

With the pieces averaged, the *cancer* part counts, so the long text no longer matches the cut one.

## References

- [TimSchopf/medical_abstracts dataset card](https://huggingface.co/datasets/TimSchopf/medical_abstracts) and [source repository](https://github.com/sebischair/Medical-Abstracts-TC-Corpus)
- Schopf, T., Braun, D. and Matthes, F. (2022). Evaluating Unsupervised Text Classification: Zero-shot and Similarity-based Approaches. *NLPIR 2022*, 6 to 15. https://doi.org/10.1145/3582768.3582795
- [pandas: groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) and [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
- [scikit-learn: multiclass and multilabel classification](https://scikit-learn.org/stable/modules/multiclass.html)
- [thenlper/gte-small model card](https://huggingface.co/thenlper/gte-small)